## check mask
1. 同一序列的mask标签是否是互斥的，如果是，可以将所有分割融合到同一个模型中
   1. mask值与标签的对应关系
   2. 标签间是否彼此互斥，能否直接融合

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
from collections import  Counter
import SimpleITK as sitk

In [2]:
# 加载单个mask数据，查看mask数据的标签值以及数据类型
# series_uid = '../data/Liver/3Dircadb1.10/MASKS_DICOM/MASKS_DICOM/bone'
# reader = sitk.ImageSeriesReader()
# filenamesDicom = reader.GetGDCMSeriesFileNames(series_uid)
# reader.SetFileNames(filenamesDicom)
# img = reader.Execute()
# imgOrignal = sitk.GetArrayFromImage(img)
# img = imgOrignal.flatten()
# counter = Counter(img)

数据类型：uint8

### 检查标签的种类
1. 检查标签的种类
2. 是否每组数据都有完全的种类
3. 是否每个标签的mask都是uint8
4. 是否每个标签的值都是255

#### 检查标签的种类

```
the category number of mask is:	47
[('bone', 20), ('skin', 20), ('liver', 20), ('portalvein', 19), ('artery', 12), ('venoussystem', 12), ('livertumor', 11), ('venacava', 8), ('gallbladder', 6), ('rightkidney', 6), ('leftkidney', 6), ('spleen', 5), ('leftsurrenalgland', 3), ('rightlung', 3), ('leftlung', 3), ('stomach', 2), ('pancreas', 2), ('lungs', 2), ('rightsurrenalgland', 2), ('livertumor02', 2), ('livertumor03', 2), ('livertumor01', 2), ('portalvein1', 1), ('uterus', 1), ('colon', 1), ('bladder', 1), ('surrenalgland', 1), ('heart', 1), ('smallintestin', 1), ('liverkyste', 1), ('rightsurretumor', 1), ('leftsurretumor', 1), ('metastasectomie', 1), ('liverkyst', 1), ('livertumor04', 1), ('livertumor07', 1), ('livertumor05', 1), ('livertumor06', 1), ('metal', 1), ('Stones', 1), ('kidneys', 1), ('biliarysystem', 1), ('livertumor1', 1), ('livertumor2', 1), ('tumor', 1), ('livertumors', 1), ('livercyst', 1)]
```

|标签|中文|数量|
|:-|-|-|
|bone|骨头|20|
|skin|皮肤|20|
|liver|肝|20|
|portalvein|门脉|19|
|artery|动脉|12|
|livertumor|肝肿瘤|11|
|venacava|腔静脉|8|
|gallbladder|胆囊|6|
|rightkidney|右肾|6|
|leftkidney|左肾|6|
|spleen|脾|5|
|leftsurrenalgland|左肾上腺|3|
|rightlung|右肺|3|
|leftlung|左肺|3|
|stomach|胃|2|
|pancreas|胰腺|2|
|lungs|肺|2|
|rightsurrenalgland|尿道腺|2|
|livertumor02|肝肿瘤02|2|
|livertumor03|肝肿瘤03|2|
|livertumor01|肝肿瘤01|2|


#### 检查每组标签的值是否都是255，数据类型是否都是uint8
1. 从如下代码显示的结果来看：
    * 数据类型都是uint8
    * 数据的标签值有的是0/1，有的是0-255

#### 检查标签是0-255的，是否是0/255的二值
1. 标签值不一定是0-255的二值
    * 标签中可能混有1，但是1或255哪个是主要的标签值还不清楚
    - [ ] 需要进一步查看是1和255表达的是同一组织还是不同组织

In [3]:
data_root = '../data/Liver'
label_cnt_dict = {}
for sub_root_name in os.listdir(data_root):
    sub_root = os.path.join(data_root, sub_root_name)
    if not os.path.isdir(sub_root):
        continue
    if '3Dircadb' not in sub_root:
        continue
    mask_in_series = os.path.join(sub_root, 'MASKS_DICOM/MASKS_DICOM')
    if not os.path.isdir(mask_in_series):
        print('mask path not exist:\t{}'.format(mask_in_series))
        continue
    labels = os.listdir(mask_in_series)
    for l in labels:
        if l in label_cnt_dict:
            label_cnt_dict[l] += 1
        else:
            label_cnt_dict[l] = 1
    print('====> {}\tlabels:\t{}'.format(mask_in_series, labels))
print(label_cnt_dict)
print('the category number of mask is:\t{}'.format(len(label_cnt_dict.keys())))
label_cnt_list = (label_cnt_dict.items())
label_cnt_list = sorted(label_cnt_list, key=lambda x:x[1], reverse=True)
print(label_cnt_list)

====> ../data/Liver/3Dircadb1.10/MASKS_DICOM/MASKS_DICOM	labels:	['bone', 'portalvein1', 'livertumor', 'skin', 'venacava', 'liver']
====> ../data/Liver/3Dircadb1.20/MASKS_DICOM/MASKS_DICOM	labels:	['bone', 'stomach', 'uterus', 'gallbladder', 'colon', 'skin', 'artery', 'venacava', 'bladder', 'spleen', 'surrenalgland', 'pancreas', 'rightkidney', 'liver', 'leftkidney', 'heart', 'portalvein', 'lungs', 'smallintestin']
====> ../data/Liver/3Dircadb1.2/MASKS_DICOM/MASKS_DICOM	labels:	['bone', 'livertumor', 'gallbladder', 'skin', 'venacava', 'liver', 'portalvein']
====> ../data/Liver/3Dircadb1.9/MASKS_DICOM/MASKS_DICOM	labels:	['bone', 'livertumor', 'venoussystem', 'liverkyste', 'gallbladder', 'skin', 'artery', 'liver', 'portalvein']
====> ../data/Liver/3Dircadb1.5/MASKS_DICOM/MASKS_DICOM	labels:	['bone', 'venoussystem', 'stomach', 'skin', 'rightsurretumor', 'artery', 'spleen', 'pancreas', 'rightkidney', 'liver', 'leftsurretumor', 'leftkidney', 'portalvein', 'lungs', 'rightsurrenalgland', 'lef

In [4]:
# 检查每组标签的值是否都是255，数据类型是否都是uint8
# data_root = '../data/Liver'
# label_cnt_dict = {}
# for sub_root_name in os.listdir(data_root):
#     sub_root = os.path.join(data_root, sub_root_name)
#     if not os.path.isdir(sub_root):
#         continue
#     if '3Dircadb' not in sub_root:
#         continue
#     mask_in_series = os.path.join(sub_root, 'MASKS_DICOM/MASKS_DICOM')
#     if not os.path.isdir(mask_in_series):
#         print('mask path not exist:\t{}'.format(mask_in_series))
#         continue
#     labels = os.listdir(mask_in_series)
#     print('====> begin processing {}'.format(mask_in_series))
#     for label in labels:
#         series_uid = os.path.join(mask_in_series, label)
#         if not os.path.isdir(series_uid):
#             print('{} mask not exist!'.format(series_uid))
#         try:
#             reader = sitk.ImageSeriesReader()
#             filenamesDicom = reader.GetGDCMSeriesFileNames(series_uid)
#             reader.SetFileNames(filenamesDicom)
#             img = reader.Execute()
#             imgOrignal = sitk.GetArrayFromImage(img)
#             max_v = imgOrignal.max()
#             print('{} data type:\t{}, max value:\t{}'.format(series_uid, imgOrignal.dtype, max_v))
#         except:
#             print('error when processing {}'.format(series_uid))
#     print('====> end processing {}\n'.format(mask_in_series))

```
====> begin processing ../data/Liver/3Dircadb1.10/MASKS_DICOM/MASKS_DICOM
../data/Liver/3Dircadb1.10/MASKS_DICOM/MASKS_DICOM/bone data type:	uint8, max value:	255
../data/Liver/3Dircadb1.10/MASKS_DICOM/MASKS_DICOM/portalvein1 data type:	uint8, max value:	255
../data/Liver/3Dircadb1.10/MASKS_DICOM/MASKS_DICOM/livertumor data type:	uint8, max value:	255
../data/Liver/3Dircadb1.10/MASKS_DICOM/MASKS_DICOM/skin data type:	uint8, max value:	255
../data/Liver/3Dircadb1.10/MASKS_DICOM/MASKS_DICOM/venacava data type:	uint8, max value:	255
../data/Liver/3Dircadb1.10/MASKS_DICOM/MASKS_DICOM/liver data type:	uint8, max value:	255
====> end processing ../data/Liver/3Dircadb1.10/MASKS_DICOM/MASKS_DICOM

====> begin processing ../data/Liver/3Dircadb1.20/MASKS_DICOM/MASKS_DICOM
../data/Liver/3Dircadb1.20/MASKS_DICOM/MASKS_DICOM/bone data type:	uint8, max value:	255
../data/Liver/3Dircadb1.20/MASKS_DICOM/MASKS_DICOM/stomach data type:	uint8, max value:	255
../data/Liver/3Dircadb1.20/MASKS_DICOM/MASKS_DICOM/uterus data type:	uint8, max value:	1
../data/Liver/3Dircadb1.20/MASKS_DICOM/MASKS_DICOM/gallbladder data type:	uint8, max value:	1
../data/Liver/3Dircadb1.20/MASKS_DICOM/MASKS_DICOM/colon data type:	uint8, max value:	255
../data/Liver/3Dircadb1.20/MASKS_DICOM/MASKS_DICOM/skin data type:	uint8, max value:	255
../data/Liver/3Dircadb1.20/MASKS_DICOM/MASKS_DICOM/artery data type:	uint8, max value:	255
../data/Liver/3Dircadb1.20/MASKS_DICOM/MASKS_DICOM/venacava data type:	uint8, max value:	255
../data/Liver/3Dircadb1.20/MASKS_DICOM/MASKS_DICOM/bladder data type:	uint8, max value:	1
../data/Liver/3Dircadb1.20/MASKS_DICOM/MASKS_DICOM/spleen data type:	uint8, max value:	255
../data/Liver/3Dircadb1.20/MASKS_DICOM/MASKS_DICOM/surrenalgland data type:	uint8, max value:	1
../data/Liver/3Dircadb1.20/MASKS_DICOM/MASKS_DICOM/pancreas data type:	uint8, max value:	1
../data/Liver/3Dircadb1.20/MASKS_DICOM/MASKS_DICOM/rightkidney data type:	uint8, max value:	255
../data/Liver/3Dircadb1.20/MASKS_DICOM/MASKS_DICOM/liver data type:	uint8, max value:	1
../data/Liver/3Dircadb1.20/MASKS_DICOM/MASKS_DICOM/leftkidney data type:	uint8, max value:	255
../data/Liver/3Dircadb1.20/MASKS_DICOM/MASKS_DICOM/heart data type:	uint8, max value:	255
../data/Liver/3Dircadb1.20/MASKS_DICOM/MASKS_DICOM/portalvein data type:	uint8, max value:	255
../data/Liver/3Dircadb1.20/MASKS_DICOM/MASKS_DICOM/lungs data type:	uint8, max value:	255
../data/Liver/3Dircadb1.20/MASKS_DICOM/MASKS_DICOM/smallintestin data type:	uint8, max value:	255
====> end processing ../data/Liver/3Dircadb1.20/MASKS_DICOM/MASKS_DICOM

====> begin processing ../data/Liver/3Dircadb1.2/MASKS_DICOM/MASKS_DICOM
../data/Liver/3Dircadb1.2/MASKS_DICOM/MASKS_DICOM/bone data type:	uint8, max value:	1
../data/Liver/3Dircadb1.2/MASKS_DICOM/MASKS_DICOM/livertumor data type:	uint8, max value:	1
../data/Liver/3Dircadb1.2/MASKS_DICOM/MASKS_DICOM/gallbladder data type:	uint8, max value:	1
../data/Liver/3Dircadb1.2/MASKS_DICOM/MASKS_DICOM/skin data type:	uint8, max value:	1
../data/Liver/3Dircadb1.2/MASKS_DICOM/MASKS_DICOM/venacava data type:	uint8, max value:	255
../data/Liver/3Dircadb1.2/MASKS_DICOM/MASKS_DICOM/liver data type:	uint8, max value:	1
../data/Liver/3Dircadb1.2/MASKS_DICOM/MASKS_DICOM/portalvein data type:	uint8, max value:	255
====> end processing ../data/Liver/3Dircadb1.2/MASKS_DICOM/MASKS_DICOM

====> begin processing ../data/Liver/3Dircadb1.9/MASKS_DICOM/MASKS_DICOM
../data/Liver/3Dircadb1.9/MASKS_DICOM/MASKS_DICOM/bone data type:	uint8, max value:	255
../data/Liver/3Dircadb1.9/MASKS_DICOM/MASKS_DICOM/livertumor data type:	uint8, max value:	255
../data/Liver/3Dircadb1.9/MASKS_DICOM/MASKS_DICOM/venoussystem data type:	uint8, max value:	255
../data/Liver/3Dircadb1.9/MASKS_DICOM/MASKS_DICOM/liverkyste data type:	uint8, max value:	255
../data/Liver/3Dircadb1.9/MASKS_DICOM/MASKS_DICOM/gallbladder data type:	uint8, max value:	255
../data/Liver/3Dircadb1.9/MASKS_DICOM/MASKS_DICOM/skin data type:	uint8, max value:	255
../data/Liver/3Dircadb1.9/MASKS_DICOM/MASKS_DICOM/artery data type:	uint8, max value:	255
../data/Liver/3Dircadb1.9/MASKS_DICOM/MASKS_DICOM/liver data type:	uint8, max value:	255
../data/Liver/3Dircadb1.9/MASKS_DICOM/MASKS_DICOM/portalvein data type:	uint8, max value:	255
====> end processing ../data/Liver/3Dircadb1.9/MASKS_DICOM/MASKS_DICOM

====> begin processing ../data/Liver/3Dircadb1.5/MASKS_DICOM/MASKS_DICOM
../data/Liver/3Dircadb1.5/MASKS_DICOM/MASKS_DICOM/bone data type:	uint8, max value:	255
../data/Liver/3Dircadb1.5/MASKS_DICOM/MASKS_DICOM/venoussystem data type:	uint8, max value:	255
../data/Liver/3Dircadb1.5/MASKS_DICOM/MASKS_DICOM/stomach data type:	uint8, max value:	255
../data/Liver/3Dircadb1.5/MASKS_DICOM/MASKS_DICOM/skin data type:	uint8, max value:	255
../data/Liver/3Dircadb1.5/MASKS_DICOM/MASKS_DICOM/rightsurretumor data type:	uint8, max value:	255
../data/Liver/3Dircadb1.5/MASKS_DICOM/MASKS_DICOM/artery data type:	uint8, max value:	255
../data/Liver/3Dircadb1.5/MASKS_DICOM/MASKS_DICOM/spleen data type:	uint8, max value:	255
../data/Liver/3Dircadb1.5/MASKS_DICOM/MASKS_DICOM/pancreas data type:	uint8, max value:	255
../data/Liver/3Dircadb1.5/MASKS_DICOM/MASKS_DICOM/rightkidney data type:	uint8, max value:	255
../data/Liver/3Dircadb1.5/MASKS_DICOM/MASKS_DICOM/liver data type:	uint8, max value:	255
../data/Liver/3Dircadb1.5/MASKS_DICOM/MASKS_DICOM/leftsurretumor data type:	uint8, max value:	255
../data/Liver/3Dircadb1.5/MASKS_DICOM/MASKS_DICOM/leftkidney data type:	uint8, max value:	255
../data/Liver/3Dircadb1.5/MASKS_DICOM/MASKS_DICOM/portalvein data type:	uint8, max value:	255
../data/Liver/3Dircadb1.5/MASKS_DICOM/MASKS_DICOM/lungs data type:	uint8, max value:	255
../data/Liver/3Dircadb1.5/MASKS_DICOM/MASKS_DICOM/rightsurrenalgland data type:	uint8, max value:	255
../data/Liver/3Dircadb1.5/MASKS_DICOM/MASKS_DICOM/leftsurrenalgland data type:	uint8, max value:	255
====> end processing ../data/Liver/3Dircadb1.5/MASKS_DICOM/MASKS_DICOM

====> begin processing ../data/Liver/3Dircadb1.14/MASKS_DICOM/MASKS_DICOM
../data/Liver/3Dircadb1.14/MASKS_DICOM/MASKS_DICOM/bone data type:	uint8, max value:	255
../data/Liver/3Dircadb1.14/MASKS_DICOM/MASKS_DICOM/venoussystem data type:	uint8, max value:	255
../data/Liver/3Dircadb1.14/MASKS_DICOM/MASKS_DICOM/skin data type:	uint8, max value:	255
../data/Liver/3Dircadb1.14/MASKS_DICOM/MASKS_DICOM/metastasectomie data type:	uint8, max value:	255
../data/Liver/3Dircadb1.14/MASKS_DICOM/MASKS_DICOM/liver data type:	uint8, max value:	255
../data/Liver/3Dircadb1.14/MASKS_DICOM/MASKS_DICOM/portalvein data type:	uint8, max value:	255
====> end processing ../data/Liver/3Dircadb1.14/MASKS_DICOM/MASKS_DICOM

====> begin processing ../data/Liver/3Dircadb1.15/MASKS_DICOM/MASKS_DICOM
../data/Liver/3Dircadb1.15/MASKS_DICOM/MASKS_DICOM/bone data type:	uint8, max value:	255
../data/Liver/3Dircadb1.15/MASKS_DICOM/MASKS_DICOM/livertumor data type:	uint8, max value:	255
../data/Liver/3Dircadb1.15/MASKS_DICOM/MASKS_DICOM/venoussystem data type:	uint8, max value:	255
../data/Liver/3Dircadb1.15/MASKS_DICOM/MASKS_DICOM/skin data type:	uint8, max value:	255
../data/Liver/3Dircadb1.15/MASKS_DICOM/MASKS_DICOM/liver data type:	uint8, max value:	255
../data/Liver/3Dircadb1.15/MASKS_DICOM/MASKS_DICOM/portalvein data type:	uint8, max value:	255
====> end processing ../data/Liver/3Dircadb1.15/MASKS_DICOM/MASKS_DICOM

====> begin processing ../data/Liver/3Dircadb1.11/MASKS_DICOM/MASKS_DICOM
../data/Liver/3Dircadb1.11/MASKS_DICOM/MASKS_DICOM/bone data type:	uint8, max value:	255
../data/Liver/3Dircadb1.11/MASKS_DICOM/MASKS_DICOM/gallbladder data type:	uint8, max value:	255
../data/Liver/3Dircadb1.11/MASKS_DICOM/MASKS_DICOM/skin data type:	uint8, max value:	1
../data/Liver/3Dircadb1.11/MASKS_DICOM/MASKS_DICOM/artery data type:	uint8, max value:	255
../data/Liver/3Dircadb1.11/MASKS_DICOM/MASKS_DICOM/venacava data type:	uint8, max value:	255
../data/Liver/3Dircadb1.11/MASKS_DICOM/MASKS_DICOM/rightkidney data type:	uint8, max value:	255
../data/Liver/3Dircadb1.11/MASKS_DICOM/MASKS_DICOM/liver data type:	uint8, max value:	255
../data/Liver/3Dircadb1.11/MASKS_DICOM/MASKS_DICOM/leftkidney data type:	uint8, max value:	255
../data/Liver/3Dircadb1.11/MASKS_DICOM/MASKS_DICOM/portalvein data type:	uint8, max value:	255
../data/Liver/3Dircadb1.11/MASKS_DICOM/MASKS_DICOM/rightsurrenalgland data type:	uint8, max value:	255
../data/Liver/3Dircadb1.11/MASKS_DICOM/MASKS_DICOM/leftsurrenalgland data type:	uint8, max value:	255
====> end processing ../data/Liver/3Dircadb1.11/MASKS_DICOM/MASKS_DICOM

====> begin processing ../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM
../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM/bone data type:	uint8, max value:	255
../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM/livertumor02 data type:	uint8, max value:	255
../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM/venoussystem data type:	uint8, max value:	255
../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM/liverkyst data type:	uint8, max value:	255
../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM/rightlung data type:	uint8, max value:	255
../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM/skin data type:	uint8, max value:	255
../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM/livertumor04 data type:	uint8, max value:	255
../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM/artery data type:	uint8, max value:	255
../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM/livertumor03 data type:	uint8, max value:	255
../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM/livertumor07 data type:	uint8, max value:	255
../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM/livertumor05 data type:	uint8, max value:	255
../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM/leftlung data type:	uint8, max value:	255
../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM/spleen data type:	uint8, max value:	255
../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM/rightkidney data type:	uint8, max value:	255
../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM/liver data type:	uint8, max value:	255
../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM/leftkidney data type:	uint8, max value:	255
../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM/livertumor06 data type:	uint8, max value:	255
../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM/portalvein data type:	uint8, max value:	255
../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM/livertumor01 data type:	uint8, max value:	255
====> end processing ../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM

====> begin processing ../data/Liver/3Dircadb1.3/MASKS_DICOM/MASKS_DICOM
../data/Liver/3Dircadb1.3/MASKS_DICOM/MASKS_DICOM/bone data type:	uint8, max value:	1
../data/Liver/3Dircadb1.3/MASKS_DICOM/MASKS_DICOM/livertumor data type:	uint8, max value:	255
../data/Liver/3Dircadb1.3/MASKS_DICOM/MASKS_DICOM/metal data type:	uint8, max value:	1
../data/Liver/3Dircadb1.3/MASKS_DICOM/MASKS_DICOM/rightlung data type:	uint8, max value:	1
../data/Liver/3Dircadb1.3/MASKS_DICOM/MASKS_DICOM/Stones data type:	uint8, max value:	255
../data/Liver/3Dircadb1.3/MASKS_DICOM/MASKS_DICOM/skin data type:	uint8, max value:	1
../data/Liver/3Dircadb1.3/MASKS_DICOM/MASKS_DICOM/kidneys data type:	uint8, max value:	255
../data/Liver/3Dircadb1.3/MASKS_DICOM/MASKS_DICOM/venacava data type:	uint8, max value:	1
../data/Liver/3Dircadb1.3/MASKS_DICOM/MASKS_DICOM/leftlung data type:	uint8, max value:	1
../data/Liver/3Dircadb1.3/MASKS_DICOM/MASKS_DICOM/biliarysystem data type:	uint8, max value:	255
../data/Liver/3Dircadb1.3/MASKS_DICOM/MASKS_DICOM/spleen data type:	uint8, max value:	1
../data/Liver/3Dircadb1.3/MASKS_DICOM/MASKS_DICOM/liver data type:	uint8, max value:	255
../data/Liver/3Dircadb1.3/MASKS_DICOM/MASKS_DICOM/portalvein data type:	uint8, max value:	1
====> end processing ../data/Liver/3Dircadb1.3/MASKS_DICOM/MASKS_DICOM

====> begin processing ../data/Liver/3Dircadb1.17/MASKS_DICOM/MASKS_DICOM
../data/Liver/3Dircadb1.17/MASKS_DICOM/MASKS_DICOM/bone data type:	uint8, max value:	255
../data/Liver/3Dircadb1.17/MASKS_DICOM/MASKS_DICOM/venoussystem data type:	uint8, max value:	255
../data/Liver/3Dircadb1.17/MASKS_DICOM/MASKS_DICOM/livertumor1 data type:	uint8, max value:	255
../data/Liver/3Dircadb1.17/MASKS_DICOM/MASKS_DICOM/skin data type:	uint8, max value:	255
../data/Liver/3Dircadb1.17/MASKS_DICOM/MASKS_DICOM/artery data type:	uint8, max value:	255
../data/Liver/3Dircadb1.17/MASKS_DICOM/MASKS_DICOM/liver data type:	uint8, max value:	255
../data/Liver/3Dircadb1.17/MASKS_DICOM/MASKS_DICOM/livertumor2 data type:	uint8, max value:	255
../data/Liver/3Dircadb1.17/MASKS_DICOM/MASKS_DICOM/portalvein data type:	uint8, max value:	255
====> end processing ../data/Liver/3Dircadb1.17/MASKS_DICOM/MASKS_DICOM

====> begin processing ../data/Liver/3Dircadb1.6/MASKS_DICOM/MASKS_DICOM
../data/Liver/3Dircadb1.6/MASKS_DICOM/MASKS_DICOM/bone data type:	uint8, max value:	255
../data/Liver/3Dircadb1.6/MASKS_DICOM/MASKS_DICOM/livertumor data type:	uint8, max value:	255
../data/Liver/3Dircadb1.6/MASKS_DICOM/MASKS_DICOM/venoussystem data type:	uint8, max value:	255
../data/Liver/3Dircadb1.6/MASKS_DICOM/MASKS_DICOM/skin data type:	uint8, max value:	255
../data/Liver/3Dircadb1.6/MASKS_DICOM/MASKS_DICOM/artery data type:	uint8, max value:	255
../data/Liver/3Dircadb1.6/MASKS_DICOM/MASKS_DICOM/rightkidney data type:	uint8, max value:	255
../data/Liver/3Dircadb1.6/MASKS_DICOM/MASKS_DICOM/liver data type:	uint8, max value:	255
../data/Liver/3Dircadb1.6/MASKS_DICOM/MASKS_DICOM/leftkidney data type:	uint8, max value:	255
../data/Liver/3Dircadb1.6/MASKS_DICOM/MASKS_DICOM/portalvein data type:	uint8, max value:	255
====> end processing ../data/Liver/3Dircadb1.6/MASKS_DICOM/MASKS_DICOM

====> begin processing ../data/Liver/3Dircadb1.4/MASKS_DICOM/MASKS_DICOM
../data/Liver/3Dircadb1.4/MASKS_DICOM/MASKS_DICOM/bone data type:	uint8, max value:	255
../data/Liver/3Dircadb1.4/MASKS_DICOM/MASKS_DICOM/livertumor data type:	uint8, max value:	255
../data/Liver/3Dircadb1.4/MASKS_DICOM/MASKS_DICOM/venoussystem data type:	uint8, max value:	255
../data/Liver/3Dircadb1.4/MASKS_DICOM/MASKS_DICOM/skin data type:	uint8, max value:	255
../data/Liver/3Dircadb1.4/MASKS_DICOM/MASKS_DICOM/artery data type:	uint8, max value:	255
../data/Liver/3Dircadb1.4/MASKS_DICOM/MASKS_DICOM/liver data type:	uint8, max value:	255
../data/Liver/3Dircadb1.4/MASKS_DICOM/MASKS_DICOM/portalvein data type:	uint8, max value:	255
====> end processing ../data/Liver/3Dircadb1.4/MASKS_DICOM/MASKS_DICOM

====> begin processing ../data/Liver/3Dircadb1.7/MASKS_DICOM/MASKS_DICOM
../data/Liver/3Dircadb1.7/MASKS_DICOM/MASKS_DICOM/bone data type:	uint8, max value:	255
../data/Liver/3Dircadb1.7/MASKS_DICOM/MASKS_DICOM/venoussystem data type:	uint8, max value:	255
../data/Liver/3Dircadb1.7/MASKS_DICOM/MASKS_DICOM/rightlung data type:	uint8, max value:	255
../data/Liver/3Dircadb1.7/MASKS_DICOM/MASKS_DICOM/skin data type:	uint8, max value:	1
../data/Liver/3Dircadb1.7/MASKS_DICOM/MASKS_DICOM/artery data type:	uint8, max value:	255
../data/Liver/3Dircadb1.7/MASKS_DICOM/MASKS_DICOM/leftlung data type:	uint8, max value:	255
../data/Liver/3Dircadb1.7/MASKS_DICOM/MASKS_DICOM/spleen data type:	uint8, max value:	255
../data/Liver/3Dircadb1.7/MASKS_DICOM/MASKS_DICOM/tumor data type:	uint8, max value:	255
../data/Liver/3Dircadb1.7/MASKS_DICOM/MASKS_DICOM/rightkidney data type:	uint8, max value:	255
../data/Liver/3Dircadb1.7/MASKS_DICOM/MASKS_DICOM/liver data type:	uint8, max value:	255
../data/Liver/3Dircadb1.7/MASKS_DICOM/MASKS_DICOM/leftkidney data type:	uint8, max value:	255
../data/Liver/3Dircadb1.7/MASKS_DICOM/MASKS_DICOM/portalvein data type:	uint8, max value:	255
../data/Liver/3Dircadb1.7/MASKS_DICOM/MASKS_DICOM/leftsurrenalgland data type:	uint8, max value:	255
====> end processing ../data/Liver/3Dircadb1.7/MASKS_DICOM/MASKS_DICOM

====> begin processing ../data/Liver/3Dircadb1.18/MASKS_DICOM/MASKS_DICOM
../data/Liver/3Dircadb1.18/MASKS_DICOM/MASKS_DICOM/bone data type:	uint8, max value:	1
../data/Liver/3Dircadb1.18/MASKS_DICOM/MASKS_DICOM/livertumor data type:	uint8, max value:	1
../data/Liver/3Dircadb1.18/MASKS_DICOM/MASKS_DICOM/gallbladder data type:	uint8, max value:	1
../data/Liver/3Dircadb1.18/MASKS_DICOM/MASKS_DICOM/skin data type:	uint8, max value:	1
../data/Liver/3Dircadb1.18/MASKS_DICOM/MASKS_DICOM/venacava data type:	uint8, max value:	255
../data/Liver/3Dircadb1.18/MASKS_DICOM/MASKS_DICOM/liver data type:	uint8, max value:	1
../data/Liver/3Dircadb1.18/MASKS_DICOM/MASKS_DICOM/portalvein data type:	uint8, max value:	255
====> end processing ../data/Liver/3Dircadb1.18/MASKS_DICOM/MASKS_DICOM

====> begin processing ../data/Liver/3Dircadb1.8/MASKS_DICOM/MASKS_DICOM
../data/Liver/3Dircadb1.8/MASKS_DICOM/MASKS_DICOM/bone data type:	uint8, max value:	255
../data/Liver/3Dircadb1.8/MASKS_DICOM/MASKS_DICOM/livertumor02 data type:	uint8, max value:	255
../data/Liver/3Dircadb1.8/MASKS_DICOM/MASKS_DICOM/venoussystem data type:	uint8, max value:	255
../data/Liver/3Dircadb1.8/MASKS_DICOM/MASKS_DICOM/skin data type:	uint8, max value:	255
../data/Liver/3Dircadb1.8/MASKS_DICOM/MASKS_DICOM/artery data type:	uint8, max value:	255
../data/Liver/3Dircadb1.8/MASKS_DICOM/MASKS_DICOM/livertumor03 data type:	uint8, max value:	255
../data/Liver/3Dircadb1.8/MASKS_DICOM/MASKS_DICOM/liver data type:	uint8, max value:	255
../data/Liver/3Dircadb1.8/MASKS_DICOM/MASKS_DICOM/portalvein data type:	uint8, max value:	255
../data/Liver/3Dircadb1.8/MASKS_DICOM/MASKS_DICOM/livertumor01 data type:	uint8, max value:	255
====> end processing ../data/Liver/3Dircadb1.8/MASKS_DICOM/MASKS_DICOM

====> begin processing ../data/Liver/3Dircadb1.19/MASKS_DICOM/MASKS_DICOM
../data/Liver/3Dircadb1.19/MASKS_DICOM/MASKS_DICOM/bone data type:	uint8, max value:	255
../data/Liver/3Dircadb1.19/MASKS_DICOM/MASKS_DICOM/livertumors data type:	uint8, max value:	255
../data/Liver/3Dircadb1.19/MASKS_DICOM/MASKS_DICOM/venoussystem data type:	uint8, max value:	255
../data/Liver/3Dircadb1.19/MASKS_DICOM/MASKS_DICOM/livercyst data type:	uint8, max value:	255
../data/Liver/3Dircadb1.19/MASKS_DICOM/MASKS_DICOM/gallbladder data type:	uint8, max value:	255
../data/Liver/3Dircadb1.19/MASKS_DICOM/MASKS_DICOM/skin data type:	uint8, max value:	255
../data/Liver/3Dircadb1.19/MASKS_DICOM/MASKS_DICOM/liver data type:	uint8, max value:	255
../data/Liver/3Dircadb1.19/MASKS_DICOM/MASKS_DICOM/portalvein data type:	uint8, max value:	255
====> end processing ../data/Liver/3Dircadb1.19/MASKS_DICOM/MASKS_DICOM

====> begin processing ../data/Liver/3Dircadb1.16/MASKS_DICOM/MASKS_DICOM
../data/Liver/3Dircadb1.16/MASKS_DICOM/MASKS_DICOM/bone data type:	uint8, max value:	255
../data/Liver/3Dircadb1.16/MASKS_DICOM/MASKS_DICOM/livertumor data type:	uint8, max value:	255
../data/Liver/3Dircadb1.16/MASKS_DICOM/MASKS_DICOM/venoussystem data type:	uint8, max value:	255
../data/Liver/3Dircadb1.16/MASKS_DICOM/MASKS_DICOM/skin data type:	uint8, max value:	255
../data/Liver/3Dircadb1.16/MASKS_DICOM/MASKS_DICOM/liver data type:	uint8, max value:	255
../data/Liver/3Dircadb1.16/MASKS_DICOM/MASKS_DICOM/portalvein data type:	uint8, max value:	255
====> end processing ../data/Liver/3Dircadb1.16/MASKS_DICOM/MASKS_DICOM

====> begin processing ../data/Liver/3Dircadb1.13/MASKS_DICOM/MASKS_DICOM
../data/Liver/3Dircadb1.13/MASKS_DICOM/MASKS_DICOM/bone data type:	uint8, max value:	255
../data/Liver/3Dircadb1.13/MASKS_DICOM/MASKS_DICOM/livertumor data type:	uint8, max value:	255
../data/Liver/3Dircadb1.13/MASKS_DICOM/MASKS_DICOM/skin data type:	uint8, max value:	255
../data/Liver/3Dircadb1.13/MASKS_DICOM/MASKS_DICOM/artery data type:	uint8, max value:	255
../data/Liver/3Dircadb1.13/MASKS_DICOM/MASKS_DICOM/venacava data type:	uint8, max value:	255
../data/Liver/3Dircadb1.13/MASKS_DICOM/MASKS_DICOM/liver data type:	uint8, max value:	255
../data/Liver/3Dircadb1.13/MASKS_DICOM/MASKS_DICOM/portalvein data type:	uint8, max value:	255
====> end processing ../data/Liver/3Dircadb1.13/MASKS_DICOM/MASKS_DICOM

====> begin processing ../data/Liver/3Dircadb1.12/MASKS_DICOM/MASKS_DICOM
../data/Liver/3Dircadb1.12/MASKS_DICOM/MASKS_DICOM/bone data type:	uint8, max value:	1
../data/Liver/3Dircadb1.12/MASKS_DICOM/MASKS_DICOM/livertumor data type:	uint8, max value:	255
../data/Liver/3Dircadb1.12/MASKS_DICOM/MASKS_DICOM/skin data type:	uint8, max value:	1
../data/Liver/3Dircadb1.12/MASKS_DICOM/MASKS_DICOM/artery data type:	uint8, max value:	1
../data/Liver/3Dircadb1.12/MASKS_DICOM/MASKS_DICOM/venacava data type:	uint8, max value:	1
../data/Liver/3Dircadb1.12/MASKS_DICOM/MASKS_DICOM/liver data type:	uint8, max value:	255
../data/Liver/3Dircadb1.12/MASKS_DICOM/MASKS_DICOM/portalvein data type:	uint8, max value:	1
====> end processing ../data/Liver/3Dircadb1.12/MASKS_DICOM/MASKS_DICOM
```

In [5]:
# # 检查标签是0-255的，是否是0/255的二值
# data_root = '../data/Liver'
# label_cnt_dict = {}
# for sub_root_name in os.listdir(data_root):
#     sub_root = os.path.join(data_root, sub_root_name)
#     if not os.path.isdir(sub_root):
#         continue
#     if '3Dircadb' not in sub_root:
#         continue
#     mask_in_series = os.path.join(sub_root, 'MASKS_DICOM/MASKS_DICOM')
#     if not os.path.isdir(mask_in_series):
#         print('mask path not exist:\t{}'.format(mask_in_series))
#         continue
#     labels = os.listdir(mask_in_series)
#     print('====> begin processing {}'.format(mask_in_series))
#     for label in labels:
#         series_uid = os.path.join(mask_in_series, label)
#         if not os.path.isdir(series_uid):
#             print('{} mask not exist!'.format(series_uid))
#         try:
#             reader = sitk.ImageSeriesReader()
#             filenamesDicom = reader.GetGDCMSeriesFileNames(series_uid)
#             reader.SetFileNames(filenamesDicom)
#             img = reader.Execute()
#             imgOrignal = sitk.GetArrayFromImage(img)
#             max_v = imgOrignal.max()
#             if max_v == 1:
#                 continue
#             img = imgOrignal.flatten()
#             counter = Counter(img)
#             print('{} data distribution:\t{}'.format(series_uid, counter))
#         except:
#             print('error when processing {}'.format(series_uid))
#     print('====> end processing {}\n'.format(mask_in_series))

====> begin processing ../data/Liver/3Dircadb1.10/MASKS_DICOM/MASKS_DICOM
../data/Liver/3Dircadb1.10/MASKS_DICOM/MASKS_DICOM/bone data distribution:	Counter({0: 31593358, 255: 388210})
../data/Liver/3Dircadb1.10/MASKS_DICOM/MASKS_DICOM/portalvein1 data distribution:	Counter({0: 31949812, 255: 31756})
../data/Liver/3Dircadb1.10/MASKS_DICOM/MASKS_DICOM/livertumor data distribution:	Counter({0: 31966587, 255: 14981})
../data/Liver/3Dircadb1.10/MASKS_DICOM/MASKS_DICOM/skin data distribution:	Counter({0: 17623656, 255: 14357912})
../data/Liver/3Dircadb1.10/MASKS_DICOM/MASKS_DICOM/venacava data distribution:	Counter({0: 31915366, 255: 66202})
../data/Liver/3Dircadb1.10/MASKS_DICOM/MASKS_DICOM/liver data distribution:	Counter({0: 30109764, 255: 1871804})
====> end processing ../data/Liver/3Dircadb1.10/MASKS_DICOM/MASKS_DICOM

====> begin processing ../data/Liver/3Dircadb1.20/MASKS_DICOM/MASKS_DICOM
../data/Liver/3Dircadb1.20/MASKS_DICOM/MASKS_DICOM/bone data distribution:	Counter({0: 57486780

../data/Liver/3Dircadb1.11/MASKS_DICOM/MASKS_DICOM/rightkidney data distribution:	Counter({0: 34431000, 255: 172008})
../data/Liver/3Dircadb1.11/MASKS_DICOM/MASKS_DICOM/liver data distribution:	Counter({0: 32910292, 255: 1692716})
../data/Liver/3Dircadb1.11/MASKS_DICOM/MASKS_DICOM/leftkidney data distribution:	Counter({0: 34433624, 255: 169384})
../data/Liver/3Dircadb1.11/MASKS_DICOM/MASKS_DICOM/portalvein data distribution:	Counter({0: 34537980, 255: 65028})
../data/Liver/3Dircadb1.11/MASKS_DICOM/MASKS_DICOM/rightsurrenalgland data distribution:	Counter({0: 34597079, 255: 5929})
../data/Liver/3Dircadb1.11/MASKS_DICOM/MASKS_DICOM/leftsurrenalgland data distribution:	Counter({0: 34597490, 255: 5518})
====> end processing ../data/Liver/3Dircadb1.11/MASKS_DICOM/MASKS_DICOM

====> begin processing ../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM
../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM/bone data distribution:	Counter({0: 33099183, 255: 717393})
../data/Liver/3Dircadb1.1/MASKS_DIC

../data/Liver/3Dircadb1.7/MASKS_DICOM/MASKS_DICOM/portalvein data distribution:	Counter({0: 39524223, 255: 59521})
../data/Liver/3Dircadb1.7/MASKS_DICOM/MASKS_DICOM/leftsurrenalgland data distribution:	Counter({0: 39576378, 255: 7366})
====> end processing ../data/Liver/3Dircadb1.7/MASKS_DICOM/MASKS_DICOM

====> begin processing ../data/Liver/3Dircadb1.18/MASKS_DICOM/MASKS_DICOM
../data/Liver/3Dircadb1.18/MASKS_DICOM/MASKS_DICOM/venacava data distribution:	Counter({0: 19353420, 255: 45236})
../data/Liver/3Dircadb1.18/MASKS_DICOM/MASKS_DICOM/portalvein data distribution:	Counter({0: 19367908, 255: 30748})
====> end processing ../data/Liver/3Dircadb1.18/MASKS_DICOM/MASKS_DICOM

====> begin processing ../data/Liver/3Dircadb1.8/MASKS_DICOM/MASKS_DICOM
../data/Liver/3Dircadb1.8/MASKS_DICOM/MASKS_DICOM/bone data distribution:	Counter({0: 31786367, 255: 719489})
../data/Liver/3Dircadb1.8/MASKS_DICOM/MASKS_DICOM/livertumor02 data distribution:	Counter({0: 32497435, 255: 8421})
../data/Liver/3D

```
====> begin processing ../data/Liver/3Dircadb1.10/MASKS_DICOM/MASKS_DICOM
../data/Liver/3Dircadb1.10/MASKS_DICOM/MASKS_DICOM/bone data distribution:	Counter({0: 31593358, 255: 388210})
../data/Liver/3Dircadb1.10/MASKS_DICOM/MASKS_DICOM/portalvein1 data distribution:	Counter({0: 31949812, 255: 31756})
../data/Liver/3Dircadb1.10/MASKS_DICOM/MASKS_DICOM/livertumor data distribution:	Counter({0: 31966587, 255: 14981})
../data/Liver/3Dircadb1.10/MASKS_DICOM/MASKS_DICOM/skin data distribution:	Counter({0: 17623656, 255: 14357912})
../data/Liver/3Dircadb1.10/MASKS_DICOM/MASKS_DICOM/venacava data distribution:	Counter({0: 31915366, 255: 66202})
../data/Liver/3Dircadb1.10/MASKS_DICOM/MASKS_DICOM/liver data distribution:	Counter({0: 30109764, 255: 1871804})
====> end processing ../data/Liver/3Dircadb1.10/MASKS_DICOM/MASKS_DICOM

====> begin processing ../data/Liver/3Dircadb1.20/MASKS_DICOM/MASKS_DICOM
../data/Liver/3Dircadb1.20/MASKS_DICOM/MASKS_DICOM/bone data distribution:	Counter({0: 57486780, 1: 1495619, 255: 1})
../data/Liver/3Dircadb1.20/MASKS_DICOM/MASKS_DICOM/stomach data distribution:	Counter({0: 58634829, 255: 347571})
../data/Liver/3Dircadb1.20/MASKS_DICOM/MASKS_DICOM/colon data distribution:	Counter({0: 58486465, 255: 495935})
../data/Liver/3Dircadb1.20/MASKS_DICOM/MASKS_DICOM/skin data distribution:	Counter({255: 30207074, 0: 28775326})
../data/Liver/3Dircadb1.20/MASKS_DICOM/MASKS_DICOM/artery data distribution:	Counter({0: 58876944, 1: 105438, 255: 18})
../data/Liver/3Dircadb1.20/MASKS_DICOM/MASKS_DICOM/venacava data distribution:	Counter({0: 58845824, 1: 136421, 255: 155})
../data/Liver/3Dircadb1.20/MASKS_DICOM/MASKS_DICOM/spleen data distribution:	Counter({0: 58839083, 255: 143317})
../data/Liver/3Dircadb1.20/MASKS_DICOM/MASKS_DICOM/rightkidney data distribution:	Counter({0: 58863561, 255: 118839})
../data/Liver/3Dircadb1.20/MASKS_DICOM/MASKS_DICOM/leftkidney data distribution:	Counter({0: 58896614, 255: 85786})
../data/Liver/3Dircadb1.20/MASKS_DICOM/MASKS_DICOM/heart data distribution:	Counter({0: 58717911, 255: 264489})
../data/Liver/3Dircadb1.20/MASKS_DICOM/MASKS_DICOM/portalvein data distribution:	Counter({0: 58928620, 255: 53780})
../data/Liver/3Dircadb1.20/MASKS_DICOM/MASKS_DICOM/lungs data distribution:	Counter({0: 57235842, 255: 1746558})
../data/Liver/3Dircadb1.20/MASKS_DICOM/MASKS_DICOM/smallintestin data distribution:	Counter({0: 58342430, 255: 639970})
====> end processing ../data/Liver/3Dircadb1.20/MASKS_DICOM/MASKS_DICOM

====> begin processing ../data/Liver/3Dircadb1.2/MASKS_DICOM/MASKS_DICOM
../data/Liver/3Dircadb1.2/MASKS_DICOM/MASKS_DICOM/venacava data distribution:	Counter({0: 45036472, 255: 52296})
../data/Liver/3Dircadb1.2/MASKS_DICOM/MASKS_DICOM/portalvein data distribution:	Counter({0: 45076756, 255: 12012})
====> end processing ../data/Liver/3Dircadb1.2/MASKS_DICOM/MASKS_DICOM

====> begin processing ../data/Liver/3Dircadb1.9/MASKS_DICOM/MASKS_DICOM
../data/Liver/3Dircadb1.9/MASKS_DICOM/MASKS_DICOM/bone data distribution:	Counter({0: 28763918, 255: 334066})
../data/Liver/3Dircadb1.9/MASKS_DICOM/MASKS_DICOM/livertumor data distribution:	Counter({0: 29058500, 255: 39484})
../data/Liver/3Dircadb1.9/MASKS_DICOM/MASKS_DICOM/venoussystem data distribution:	Counter({0: 28974154, 255: 123830})
../data/Liver/3Dircadb1.9/MASKS_DICOM/MASKS_DICOM/liverkyste data distribution:	Counter({0: 29094597, 255: 3387})
../data/Liver/3Dircadb1.9/MASKS_DICOM/MASKS_DICOM/gallbladder data distribution:	Counter({0: 29081443, 255: 16541})
../data/Liver/3Dircadb1.9/MASKS_DICOM/MASKS_DICOM/skin data distribution:	Counter({0: 15790725, 255: 13307259})
../data/Liver/3Dircadb1.9/MASKS_DICOM/MASKS_DICOM/artery data distribution:	Counter({0: 28927774, 255: 170210})
../data/Liver/3Dircadb1.9/MASKS_DICOM/MASKS_DICOM/liver data distribution:	Counter({0: 27832564, 255: 1265420})
../data/Liver/3Dircadb1.9/MASKS_DICOM/MASKS_DICOM/portalvein data distribution:	Counter({0: 29039269, 255: 58715})
====> end processing ../data/Liver/3Dircadb1.9/MASKS_DICOM/MASKS_DICOM

====> begin processing ../data/Liver/3Dircadb1.5/MASKS_DICOM/MASKS_DICOM
../data/Liver/3Dircadb1.5/MASKS_DICOM/MASKS_DICOM/bone data distribution:	Counter({0: 35622363, 255: 815653})
../data/Liver/3Dircadb1.5/MASKS_DICOM/MASKS_DICOM/venoussystem data distribution:	Counter({0: 36335325, 255: 102691})
../data/Liver/3Dircadb1.5/MASKS_DICOM/MASKS_DICOM/stomach data distribution:	Counter({0: 35902961, 255: 535055})
../data/Liver/3Dircadb1.5/MASKS_DICOM/MASKS_DICOM/skin data distribution:	Counter({255: 19211122, 0: 17226894})
../data/Liver/3Dircadb1.5/MASKS_DICOM/MASKS_DICOM/rightsurretumor data distribution:	Counter({0: 36435662, 255: 2354})
../data/Liver/3Dircadb1.5/MASKS_DICOM/MASKS_DICOM/artery data distribution:	Counter({0: 36322311, 255: 115705})
../data/Liver/3Dircadb1.5/MASKS_DICOM/MASKS_DICOM/spleen data distribution:	Counter({0: 36237732, 255: 200284})
../data/Liver/3Dircadb1.5/MASKS_DICOM/MASKS_DICOM/pancreas data distribution:	Counter({0: 36337538, 255: 100478})
../data/Liver/3Dircadb1.5/MASKS_DICOM/MASKS_DICOM/rightkidney data distribution:	Counter({0: 36197270, 255: 240746})
../data/Liver/3Dircadb1.5/MASKS_DICOM/MASKS_DICOM/liver data distribution:	Counter({0: 34313511, 255: 2124505})
../data/Liver/3Dircadb1.5/MASKS_DICOM/MASKS_DICOM/leftsurretumor data distribution:	Counter({0: 36434640, 255: 3376})
../data/Liver/3Dircadb1.5/MASKS_DICOM/MASKS_DICOM/leftkidney data distribution:	Counter({0: 36216316, 255: 221700})
../data/Liver/3Dircadb1.5/MASKS_DICOM/MASKS_DICOM/portalvein data distribution:	Counter({0: 36376242, 255: 61774})
../data/Liver/3Dircadb1.5/MASKS_DICOM/MASKS_DICOM/lungs data distribution:	Counter({0: 34979179, 255: 1458837})
../data/Liver/3Dircadb1.5/MASKS_DICOM/MASKS_DICOM/rightsurrenalgland data distribution:	Counter({0: 36428897, 255: 9119})
../data/Liver/3Dircadb1.5/MASKS_DICOM/MASKS_DICOM/leftsurrenalgland data distribution:	Counter({0: 36427011, 255: 11005})
====> end processing ../data/Liver/3Dircadb1.5/MASKS_DICOM/MASKS_DICOM

====> begin processing ../data/Liver/3Dircadb1.14/MASKS_DICOM/MASKS_DICOM
../data/Liver/3Dircadb1.14/MASKS_DICOM/MASKS_DICOM/bone data distribution:	Counter({0: 29214146, 255: 408126})
../data/Liver/3Dircadb1.14/MASKS_DICOM/MASKS_DICOM/venoussystem data distribution:	Counter({0: 29525543, 255: 96729})
../data/Liver/3Dircadb1.14/MASKS_DICOM/MASKS_DICOM/skin data distribution:	Counter({0: 17757428, 255: 11864844})
../data/Liver/3Dircadb1.14/MASKS_DICOM/MASKS_DICOM/metastasectomie data distribution:	Counter({0: 29589545, 255: 32727})
../data/Liver/3Dircadb1.14/MASKS_DICOM/MASKS_DICOM/liver data distribution:	Counter({0: 27988631, 255: 1633641})
../data/Liver/3Dircadb1.14/MASKS_DICOM/MASKS_DICOM/portalvein data distribution:	Counter({0: 29569564, 255: 52606, 1: 102})
====> end processing ../data/Liver/3Dircadb1.14/MASKS_DICOM/MASKS_DICOM

====> begin processing ../data/Liver/3Dircadb1.15/MASKS_DICOM/MASKS_DICOM
../data/Liver/3Dircadb1.15/MASKS_DICOM/MASKS_DICOM/bone data distribution:	Counter({0: 32342347, 255: 425653})
../data/Liver/3Dircadb1.15/MASKS_DICOM/MASKS_DICOM/livertumor data distribution:	Counter({0: 32766362, 255: 1638})
../data/Liver/3Dircadb1.15/MASKS_DICOM/MASKS_DICOM/venoussystem data distribution:	Counter({0: 32701046, 255: 66954})
../data/Liver/3Dircadb1.15/MASKS_DICOM/MASKS_DICOM/skin data distribution:	Counter({0: 20633282, 255: 12134718})
../data/Liver/3Dircadb1.15/MASKS_DICOM/MASKS_DICOM/liver data distribution:	Counter({0: 31378428, 255: 1389572})
../data/Liver/3Dircadb1.15/MASKS_DICOM/MASKS_DICOM/portalvein data distribution:	Counter({0: 32719655, 255: 48345})
====> end processing ../data/Liver/3Dircadb1.15/MASKS_DICOM/MASKS_DICOM

====> begin processing ../data/Liver/3Dircadb1.11/MASKS_DICOM/MASKS_DICOM
../data/Liver/3Dircadb1.11/MASKS_DICOM/MASKS_DICOM/bone data distribution:	Counter({0: 34047352, 255: 555656})
../data/Liver/3Dircadb1.11/MASKS_DICOM/MASKS_DICOM/gallbladder data distribution:	Counter({0: 34553763, 255: 49245})
../data/Liver/3Dircadb1.11/MASKS_DICOM/MASKS_DICOM/artery data distribution:	Counter({0: 34500447, 255: 102560, 1: 1})
../data/Liver/3Dircadb1.11/MASKS_DICOM/MASKS_DICOM/venacava data distribution:	Counter({0: 34499838, 255: 103170})
../data/Liver/3Dircadb1.11/MASKS_DICOM/MASKS_DICOM/rightkidney data distribution:	Counter({0: 34431000, 255: 172008})
../data/Liver/3Dircadb1.11/MASKS_DICOM/MASKS_DICOM/liver data distribution:	Counter({0: 32910292, 255: 1692716})
../data/Liver/3Dircadb1.11/MASKS_DICOM/MASKS_DICOM/leftkidney data distribution:	Counter({0: 34433624, 255: 169384})
../data/Liver/3Dircadb1.11/MASKS_DICOM/MASKS_DICOM/portalvein data distribution:	Counter({0: 34537980, 255: 65028})
../data/Liver/3Dircadb1.11/MASKS_DICOM/MASKS_DICOM/rightsurrenalgland data distribution:	Counter({0: 34597079, 255: 5929})
../data/Liver/3Dircadb1.11/MASKS_DICOM/MASKS_DICOM/leftsurrenalgland data distribution:	Counter({0: 34597490, 255: 5518})
====> end processing ../data/Liver/3Dircadb1.11/MASKS_DICOM/MASKS_DICOM

====> begin processing ../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM
../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM/bone data distribution:	Counter({0: 33099183, 255: 717393})
../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM/livertumor02 data distribution:	Counter({0: 33802112, 255: 14464})
../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM/venoussystem data distribution:	Counter({0: 33673396, 255: 143180})
../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM/liverkyst data distribution:	Counter({0: 33816468, 255: 108})
../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM/rightlung data distribution:	Counter({0: 33135856, 255: 680720})
../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM/skin data distribution:	Counter({255: 19301754, 0: 14514822})
../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM/livertumor04 data distribution:	Counter({0: 33811717, 255: 4859})
../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM/artery data distribution:	Counter({0: 33701653, 255: 114923})
../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM/livertumor03 data distribution:	Counter({0: 33809384, 255: 7192})
../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM/livertumor07 data distribution:	Counter({0: 33811352, 255: 5224})
../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM/livertumor05 data distribution:	Counter({0: 33797150, 255: 19426})
../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM/leftlung data distribution:	Counter({0: 33235227, 255: 581349})
../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM/spleen data distribution:	Counter({0: 33593803, 255: 222773})
../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM/rightkidney data distribution:	Counter({0: 33601218, 255: 215358})
../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM/liver data distribution:	Counter({0: 30951445, 255: 2865131})
../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM/leftkidney data distribution:	Counter({0: 33536171, 255: 280405})
../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM/livertumor06 data distribution:	Counter({0: 33815810, 255: 766})
../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM/portalvein data distribution:	Counter({0: 33713043, 255: 103533})
../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM/livertumor01 data distribution:	Counter({0: 33654239, 255: 162337})
====> end processing ../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM

====> begin processing ../data/Liver/3Dircadb1.3/MASKS_DICOM/MASKS_DICOM
../data/Liver/3Dircadb1.3/MASKS_DICOM/MASKS_DICOM/livertumor data distribution:	Counter({0: 52414548, 255: 14252})
../data/Liver/3Dircadb1.3/MASKS_DICOM/MASKS_DICOM/Stones data distribution:	Counter({0: 52428748, 255: 52})
../data/Liver/3Dircadb1.3/MASKS_DICOM/MASKS_DICOM/kidneys data distribution:	Counter({0: 51959386, 255: 469414})
../data/Liver/3Dircadb1.3/MASKS_DICOM/MASKS_DICOM/biliarysystem data distribution:	Counter({0: 52312860, 255: 115940})
../data/Liver/3Dircadb1.3/MASKS_DICOM/MASKS_DICOM/liver data distribution:	Counter({0: 50053721, 255: 2375079})
====> end processing ../data/Liver/3Dircadb1.3/MASKS_DICOM/MASKS_DICOM

====> begin processing ../data/Liver/3Dircadb1.17/MASKS_DICOM/MASKS_DICOM
../data/Liver/3Dircadb1.17/MASKS_DICOM/MASKS_DICOM/bone data distribution:	Counter({0: 30717730, 255: 477406})
../data/Liver/3Dircadb1.17/MASKS_DICOM/MASKS_DICOM/venoussystem data distribution:	Counter({0: 31070271, 255: 124865})
../data/Liver/3Dircadb1.17/MASKS_DICOM/MASKS_DICOM/livertumor1 data distribution:	Counter({0: 31098175, 255: 96961})
../data/Liver/3Dircadb1.17/MASKS_DICOM/MASKS_DICOM/skin data distribution:	Counter({255: 15984710, 0: 15210426})
../data/Liver/3Dircadb1.17/MASKS_DICOM/MASKS_DICOM/artery data distribution:	Counter({0: 31073657, 255: 121479})
../data/Liver/3Dircadb1.17/MASKS_DICOM/MASKS_DICOM/liver data distribution:	Counter({0: 29088639, 255: 2106497})
../data/Liver/3Dircadb1.17/MASKS_DICOM/MASKS_DICOM/livertumor2 data distribution:	Counter({0: 31106107, 255: 89029})
../data/Liver/3Dircadb1.17/MASKS_DICOM/MASKS_DICOM/portalvein data distribution:	Counter({0: 31141390, 255: 53746})
====> end processing ../data/Liver/3Dircadb1.17/MASKS_DICOM/MASKS_DICOM

====> begin processing ../data/Liver/3Dircadb1.6/MASKS_DICOM/MASKS_DICOM
../data/Liver/3Dircadb1.6/MASKS_DICOM/MASKS_DICOM/bone data distribution:	Counter({0: 34835128, 255: 554312})
../data/Liver/3Dircadb1.6/MASKS_DICOM/MASKS_DICOM/livertumor data distribution:	Counter({0: 34954719, 255: 434721})
../data/Liver/3Dircadb1.6/MASKS_DICOM/MASKS_DICOM/venoussystem data distribution:	Counter({0: 35229345, 255: 160095})
../data/Liver/3Dircadb1.6/MASKS_DICOM/MASKS_DICOM/skin data distribution:	Counter({255: 20434422, 0: 14955018})
../data/Liver/3Dircadb1.6/MASKS_DICOM/MASKS_DICOM/artery data distribution:	Counter({0: 35245683, 255: 143757})
../data/Liver/3Dircadb1.6/MASKS_DICOM/MASKS_DICOM/rightkidney data distribution:	Counter({0: 35230125, 255: 159315})
../data/Liver/3Dircadb1.6/MASKS_DICOM/MASKS_DICOM/liver data distribution:	Counter({0: 33560947, 255: 1828493})
../data/Liver/3Dircadb1.6/MASKS_DICOM/MASKS_DICOM/leftkidney data distribution:	Counter({0: 35215254, 255: 174186})
../data/Liver/3Dircadb1.6/MASKS_DICOM/MASKS_DICOM/portalvein data distribution:	Counter({0: 35293356, 255: 96084})
====> end processing ../data/Liver/3Dircadb1.6/MASKS_DICOM/MASKS_DICOM

====> begin processing ../data/Liver/3Dircadb1.4/MASKS_DICOM/MASKS_DICOM
../data/Liver/3Dircadb1.4/MASKS_DICOM/MASKS_DICOM/bone data distribution:	Counter({0: 23481161, 255: 373943})
../data/Liver/3Dircadb1.4/MASKS_DICOM/MASKS_DICOM/livertumor data distribution:	Counter({0: 23848368, 255: 6736})
../data/Liver/3Dircadb1.4/MASKS_DICOM/MASKS_DICOM/venoussystem data distribution:	Counter({0: 23711091, 255: 144013})
../data/Liver/3Dircadb1.4/MASKS_DICOM/MASKS_DICOM/skin data distribution:	Counter({0: 13419428, 255: 10435676})
../data/Liver/3Dircadb1.4/MASKS_DICOM/MASKS_DICOM/artery data distribution:	Counter({0: 23790891, 255: 64213})
../data/Liver/3Dircadb1.4/MASKS_DICOM/MASKS_DICOM/liver data distribution:	Counter({0: 22722677, 255: 1132427})
../data/Liver/3Dircadb1.4/MASKS_DICOM/MASKS_DICOM/portalvein data distribution:	Counter({0: 23779874, 255: 75230})
====> end processing ../data/Liver/3Dircadb1.4/MASKS_DICOM/MASKS_DICOM

====> begin processing ../data/Liver/3Dircadb1.7/MASKS_DICOM/MASKS_DICOM
../data/Liver/3Dircadb1.7/MASKS_DICOM/MASKS_DICOM/bone data distribution:	Counter({0: 38732762, 255: 850982})
../data/Liver/3Dircadb1.7/MASKS_DICOM/MASKS_DICOM/venoussystem data distribution:	Counter({0: 39417962, 255: 165782})
../data/Liver/3Dircadb1.7/MASKS_DICOM/MASKS_DICOM/rightlung data distribution:	Counter({0: 38842311, 255: 741433})
../data/Liver/3Dircadb1.7/MASKS_DICOM/MASKS_DICOM/artery data distribution:	Counter({0: 39485115, 255: 98629})
../data/Liver/3Dircadb1.7/MASKS_DICOM/MASKS_DICOM/leftlung data distribution:	Counter({0: 38763066, 255: 820678})
../data/Liver/3Dircadb1.7/MASKS_DICOM/MASKS_DICOM/spleen data distribution:	Counter({0: 39372028, 255: 211716})
../data/Liver/3Dircadb1.7/MASKS_DICOM/MASKS_DICOM/tumor data distribution:	Counter({0: 38900188, 255: 683556})
../data/Liver/3Dircadb1.7/MASKS_DICOM/MASKS_DICOM/rightkidney data distribution:	Counter({0: 39442226, 255: 141518})
../data/Liver/3Dircadb1.7/MASKS_DICOM/MASKS_DICOM/liver data distribution:	Counter({0: 38121800, 255: 1461944})
../data/Liver/3Dircadb1.7/MASKS_DICOM/MASKS_DICOM/leftkidney data distribution:	Counter({0: 39431771, 255: 151973})
../data/Liver/3Dircadb1.7/MASKS_DICOM/MASKS_DICOM/portalvein data distribution:	Counter({0: 39524223, 255: 59521})
../data/Liver/3Dircadb1.7/MASKS_DICOM/MASKS_DICOM/leftsurrenalgland data distribution:	Counter({0: 39576378, 255: 7366})
====> end processing ../data/Liver/3Dircadb1.7/MASKS_DICOM/MASKS_DICOM

====> begin processing ../data/Liver/3Dircadb1.18/MASKS_DICOM/MASKS_DICOM
../data/Liver/3Dircadb1.18/MASKS_DICOM/MASKS_DICOM/venacava data distribution:	Counter({0: 19353420, 255: 45236})
../data/Liver/3Dircadb1.18/MASKS_DICOM/MASKS_DICOM/portalvein data distribution:	Counter({0: 19367908, 255: 30748})
====> end processing ../data/Liver/3Dircadb1.18/MASKS_DICOM/MASKS_DICOM

====> begin processing ../data/Liver/3Dircadb1.8/MASKS_DICOM/MASKS_DICOM
../data/Liver/3Dircadb1.8/MASKS_DICOM/MASKS_DICOM/bone data distribution:	Counter({0: 31786367, 255: 719489})
../data/Liver/3Dircadb1.8/MASKS_DICOM/MASKS_DICOM/livertumor02 data distribution:	Counter({0: 32497435, 255: 8421})
../data/Liver/3Dircadb1.8/MASKS_DICOM/MASKS_DICOM/venoussystem data distribution:	Counter({0: 32120666, 255: 385190})
../data/Liver/3Dircadb1.8/MASKS_DICOM/MASKS_DICOM/skin data distribution:	Counter({255: 16913164, 0: 15592692})
../data/Liver/3Dircadb1.8/MASKS_DICOM/MASKS_DICOM/artery data distribution:	Counter({0: 32385585, 255: 120271})
../data/Liver/3Dircadb1.8/MASKS_DICOM/MASKS_DICOM/livertumor03 data distribution:	Counter({0: 32495089, 255: 10767})
../data/Liver/3Dircadb1.8/MASKS_DICOM/MASKS_DICOM/liver data distribution:	Counter({0: 29290766, 255: 3215090})
../data/Liver/3Dircadb1.8/MASKS_DICOM/MASKS_DICOM/portalvein data distribution:	Counter({0: 32451690, 255: 54166})
../data/Liver/3Dircadb1.8/MASKS_DICOM/MASKS_DICOM/livertumor01 data distribution:	Counter({0: 32504726, 255: 1129, 1: 1})
====> end processing ../data/Liver/3Dircadb1.8/MASKS_DICOM/MASKS_DICOM

====> begin processing ../data/Liver/3Dircadb1.19/MASKS_DICOM/MASKS_DICOM
../data/Liver/3Dircadb1.19/MASKS_DICOM/MASKS_DICOM/bone data distribution:	Counter({0: 31669284, 255: 836572})
../data/Liver/3Dircadb1.19/MASKS_DICOM/MASKS_DICOM/livertumors data distribution:	Counter({0: 32464440, 255: 41416})
../data/Liver/3Dircadb1.19/MASKS_DICOM/MASKS_DICOM/venoussystem data distribution:	Counter({0: 32418800, 255: 87056})
../data/Liver/3Dircadb1.19/MASKS_DICOM/MASKS_DICOM/livercyst data distribution:	Counter({0: 32504441, 255: 1415})
../data/Liver/3Dircadb1.19/MASKS_DICOM/MASKS_DICOM/gallbladder data distribution:	Counter({0: 32494909, 255: 10947})
../data/Liver/3Dircadb1.19/MASKS_DICOM/MASKS_DICOM/skin data distribution:	Counter({0: 19964256, 255: 12541600})
../data/Liver/3Dircadb1.19/MASKS_DICOM/MASKS_DICOM/liver data distribution:	Counter({0: 31922648, 255: 583208})
../data/Liver/3Dircadb1.19/MASKS_DICOM/MASKS_DICOM/portalvein data distribution:	Counter({0: 32477422, 255: 28434})
====> end processing ../data/Liver/3Dircadb1.19/MASKS_DICOM/MASKS_DICOM

====> begin processing ../data/Liver/3Dircadb1.16/MASKS_DICOM/MASKS_DICOM
../data/Liver/3Dircadb1.16/MASKS_DICOM/MASKS_DICOM/bone data distribution:	Counter({0: 39943998, 255: 688322})
../data/Liver/3Dircadb1.16/MASKS_DICOM/MASKS_DICOM/livertumor data distribution:	Counter({0: 40624983, 255: 7337})
../data/Liver/3Dircadb1.16/MASKS_DICOM/MASKS_DICOM/venoussystem data distribution:	Counter({0: 40503476, 255: 128844})
../data/Liver/3Dircadb1.16/MASKS_DICOM/MASKS_DICOM/skin data distribution:	Counter({255: 26557113, 0: 14075207})
../data/Liver/3Dircadb1.16/MASKS_DICOM/MASKS_DICOM/liver data distribution:	Counter({0: 37915135, 255: 2717185})
../data/Liver/3Dircadb1.16/MASKS_DICOM/MASKS_DICOM/portalvein data distribution:	Counter({0: 40554900, 255: 77420})
====> end processing ../data/Liver/3Dircadb1.16/MASKS_DICOM/MASKS_DICOM

====> begin processing ../data/Liver/3Dircadb1.13/MASKS_DICOM/MASKS_DICOM
../data/Liver/3Dircadb1.13/MASKS_DICOM/MASKS_DICOM/bone data distribution:	Counter({0: 31513534, 255: 468034})
../data/Liver/3Dircadb1.13/MASKS_DICOM/MASKS_DICOM/livertumor data distribution:	Counter({0: 31859088, 255: 122480})
../data/Liver/3Dircadb1.13/MASKS_DICOM/MASKS_DICOM/skin data distribution:	Counter({255: 18585323, 0: 13396245})
../data/Liver/3Dircadb1.13/MASKS_DICOM/MASKS_DICOM/artery data distribution:	Counter({0: 31853236, 255: 128332})
../data/Liver/3Dircadb1.13/MASKS_DICOM/MASKS_DICOM/venacava data distribution:	Counter({0: 31849155, 255: 132413})
../data/Liver/3Dircadb1.13/MASKS_DICOM/MASKS_DICOM/liver data distribution:	Counter({0: 29918459, 255: 2063109})
../data/Liver/3Dircadb1.13/MASKS_DICOM/MASKS_DICOM/portalvein data distribution:	Counter({0: 31887423, 255: 94145})
====> end processing ../data/Liver/3Dircadb1.13/MASKS_DICOM/MASKS_DICOM

====> begin processing ../data/Liver/3Dircadb1.12/MASKS_DICOM/MASKS_DICOM
../data/Liver/3Dircadb1.12/MASKS_DICOM/MASKS_DICOM/livertumor data distribution:	Counter({0: 67808454, 255: 348986})
../data/Liver/3Dircadb1.12/MASKS_DICOM/MASKS_DICOM/liver data distribution:	Counter({0: 64816007, 1: 3341429, 255: 4})
====> end processing ../data/Liver/3Dircadb1.12/MASKS_DICOM/MASKS_DICOM
```